# FundAnalyzer — 公募基金智能分析工具

基于公开数据的公募基金全维度分析工具，覆盖业绩归因、风险评估、持仓分析、组合优化。

---
**数据来源**: 天天基金 (通过 akshare 免费获取)
**作者**: FundAnalyzer

## 1. 环境准备与导入

In [ ]:
# 安装依赖 (首次运行需取消注释)
# !pip install pandas numpy scipy akshare matplotlib -q

import sys
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'PingFang SC']
plt.rcParams['axes.unicode_minus'] = False

# 导入 FundAnalyzer
sys.path.insert(0, os.path.abspath('.'))
from fund_analyzer import fetcher, analyzer, portfolio, reporter

print('环境就绪!')

## 2. 搜索基金

按名称或代码搜索公募基金。

In [ ]:
# 搜索沪深300相关基金
results = fetcher.search_fund('沪深300')
print(f'找到 {len(results)} 只基金\n')

df_results = pd.DataFrame(results)
display_cols = ['code', 'name', 'type', 'nav', 'daily_return']
available = [c for c in display_cols if c in df_results.columns]
df_results[available].head(10)

## 3. 获取基金基本信息

以易方达沪深300ETF联接A (110020) 为例。

In [ ]:
fund_code = '110020'
info = fetcher.get_fund_info(fund_code)

print(f'基金代码: {info.get("code", fund_code)}')
print(f'基金名称: {info.get("name", "N/A")}')
print(f'基金类型: {info.get("type", "N/A")}')
print(f'管理人: {info.get("manager_company", "N/A")}')
print(f'成立日期: {info.get("inception_date", "N/A")}')
print(f'最新规模: {info.get("latest_size", "N/A")}')
print(f'管理费: {info.get("management_fee", "N/A")}')

## 4. 获取历史净值

获取最近一年的日频净值数据。

In [ ]:
nav_df = fetcher.get_fund_nav(fund_code)
print(f'数据区间: {nav_df["date"].iloc[0]} ~ {nav_df["date"].iloc[-1]}')
print(f'数据条数: {len(nav_df)}\n')
nav_df.head()

## 5. 净值趋势可视化

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# 单位净值
axes[0].plot(nav_df['date'], nav_df['nav'], linewidth=1.5, color='#1f77b4')
axes[0].set_title(f'{info.get("name", fund_code)} — 单位净值走势', fontsize=14, fontweight='bold')
axes[0].set_ylabel('单位净值', fontsize=12)
axes[0].grid(True, alpha=0.3)

# 累计净值
axes[1].plot(nav_df['date'], nav_df['acc_nav'], linewidth=1.5, color='#ff7f0e')
axes[1].set_title(f'{info.get("name", fund_code)} — 累计净值走势', fontsize=14, fontweight='bold')
axes[1].set_ylabel('累计净值', fontsize=12)
axes[1].set_xlabel('日期', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 日收益率分布

In [ ]:
returns = nav_df['nav'].pct_change().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 收益率直方图
axes[0].hist(returns, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='零收益线')
axes[0].set_title('日收益率分布', fontsize=14, fontweight='bold')
axes[0].set_xlabel('日收益率', fontsize=12)
axes[0].set_ylabel('频次', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Q-Q 图 (正态性检验)
from scipy import stats
stats.probplot(returns, dist='norm', plot=axes[1])
axes[1].set_title('正态 Q-Q 图', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'收益率均值: {returns.mean()*100:.4f}%')
print(f'收益率标准差: {returns.std()*100:.4f}%')
print(f'偏度: {returns.skew():.4f}')
print(f'峰度: {returns.kurtosis():.4f}')

## 7. 业绩指标计算

计算核心业绩与风险指标。

In [ ]:
nav = nav_df['nav']

metrics = {
    '年化收益率 (CAGR)': f'{analyzer.annualized_return(nav) * 100:.2f}%',
    '年化波动率': f'{analyzer.annualized_volatility(returns) * 100:.2f}%',
    '最大回撤': f'{analyzer.max_drawdown(nav) * 100:.2f}%',
    '夏普比率': f'{analyzer.sharpe_ratio(returns):.4f}',
    '索提诺比率': f'{analyzer.sortino_ratio(returns):.4f}',
    '卡尔玛比率': f'{analyzer.calmar_ratio(returns):.4f}',
    '胜率': f'{analyzer.win_rate(returns) * 100:.2f}%',
    '盈亏比': f'{analyzer.profit_factor(returns):.2f}',
    'VaR (95%, 历史)': f'{analyzer.var_historical(returns) * 100:.2f}%',
    'VaR (95%, 参数)': f'{analyzer.var_parametric(returns) * 100:.2f}%',
    'CVaR (95%)': f'{analyzer.conditional_var(returns) * 100:.2f}%',
    '最大连续亏损天数': analyzer.consecutive_losses(returns),
}

pd.DataFrame(list(metrics.items()), columns=['指标', '数值'])

## 8. 区间收益分析

In [ ]:
period_returns = fetcher.get_fund_returns(fund_code)
label_map = {
    '1w': '近1周', '1m': '近1月', '3m': '近3月', '6m': '近6月',
    '1y': '近1年', '2y': '近2年', '3y': '近3年',
    'ytd': '今年来', 'since_inception': '成立来'
}

period_data = []
for k, label in label_map.items():
    if k in period_returns:
        period_data.append({'区间': label, '收益率': f'{period_returns[k] * 100:.2f}%'})

pd.DataFrame(period_data)

## 9. 回撤分析

最大回撤 (Max Drawdown) 是衡量基金下行风险的核心指标。

In [ ]:
dd_series = analyzer.drawdown_series(nav)
max_dd = analyzer.max_drawdown(nav)
max_dd_date = nav_df.loc[dd_series.idxmax(), 'date'] if dd_series.idxmax() < len(nav_df) else ''

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# 净值
axes[0].plot(nav_df['date'], nav, linewidth=1.5, color='#1f77b4')
axes[0].fill_between(nav_df['date'], nav.min(), nav, alpha=0.1, color='#1f77b4')
axes[0].set_title('净值曲线', fontsize=14, fontweight='bold')
axes[0].set_ylabel('净值', fontsize=12)
axes[0].grid(True, alpha=0.3)

# 回撤
axes[1].fill_between(nav_df['date'], 0, dd_series * 100, color='crimson', alpha=0.6)
axes[1].axhline(y=max_dd * 100, color='darkred', linestyle='--', linewidth=1.5,
                label=f'最大回撤: {max_dd * 100:.2f}%')
axes[1].set_title('回撤曲线', fontsize=14, fontweight='bold')
axes[1].set_ylabel('回撤 (%)', fontsize=12)
axes[1].set_xlabel('日期', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'最大回撤: {max_dd * 100:.2f}%')
print(f'最大回撤发生日期: {max_dd_date}')

## 10. 多基金对比

In [ ]:
# 对比三只指数基金
codes_to_compare = ['110020', '000311', '161017']  # 沪深300, 中证500增强, 富国中证500

fig, ax = plt.subplots(figsize=(14, 6))

for code in codes_to_compare:
    df = fetcher.get_fund_nav(code)
    if df.empty or 'nav' not in df.columns:
        continue
    # 归一化到 100
    norm_nav = df['nav'] / df['nav'].iloc[0] * 100
    info = fetcher.get_fund_info(code)
    name = info.get('name', code)[:15]
    ax.plot(df['date'], norm_nav, linewidth=1.5, label=name)

ax.set_title('多基金净值走势对比 (归一化)', fontsize=14, fontweight='bold')
ax.set_ylabel('归一化净值 (基准=100)', fontsize=12)
ax.set_xlabel('日期', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. 持仓分析

分析主动管理型基金的前十大持仓。

In [ ]:
active_code = '110022'  # 易方达消费行业股票
holdings = fetcher.get_fund_holdings(active_code)
info_active = fetcher.get_fund_info(active_code)

print(f'{info_active.get("name", active_code)} — 前十大持仓\n')

holdings_data = []
for h in holdings[:10]:
    if 'stock_name' in h:
        holdings_data.append({
            '股票名称': h['stock_name'],
            '代码': h.get('stock_code', ''),
            '占净值比例': f"{h['ratio'] * 100:.2f}%" if h.get('ratio') else 'N/A'
        })

df_holdings = pd.DataFrame(holdings_data)
df_holdings

## 12. 投资组合优化

用三种不同风格的基金构建投资组合，寻找最优配置。

In [ ]:
# 三只不同风格的基金
pf_codes = ['110020', '110022', '005827']
# 110020 - 易方达沪深300ETF联接A (大盘)
# 110022 - 易方达消费行业股票 (消费)
# 005827 - 易方达蓝筹精选 (混合)

returns_dict = {}
names = []
for code in pf_codes:
    df = fetcher.get_fund_nav(code)
    if df.empty:
        continue
    info = fetcher.get_fund_info(code)
    name = info.get('name', code)[:12]
    returns_dict[name] = df['nav'].pct_change().dropna()
    names.append(f'{code} - {name}')

returns_df = pd.DataFrame(returns_dict)
print('组合包含的基金:')
for n in names:
    print(f'  {n}')

In [ ]:
# 蒙特卡洛模拟
mc_result = portfolio.monte_carlo_simulation(returns_df, n_portfolios=8000)

mc_returns = [s['return'] for s in mc_result['simulations']]
mc_vols = [s['vol'] for s in mc_result['simulations']]
mc_sharpes = [s['sharpe'] for s in mc_result['simulations']]

fig, ax = plt.subplots(figsize=(12, 7))

# 用颜色表示夏普比率
sc = ax.scatter(mc_vols, mc_returns, c=mc_sharpes, cmap='viridis',
                alpha=0.5, s=15, edgecolors='none')

# 标记最优组合
best = mc_result['best_sharpe']
min_v = mc_result['min_vol']

ax.scatter(best['vol'], best['return'], c='red', marker='*', s=300,
           label=f"最优夏普 (SR={best['sharpe']:.2f})", edgecolors='black', linewidth=1)
ax.scatter(min_v['vol'], min_v['return'], c='blue', marker='D', s=200,
           label=f"最小波动 (Vol={min_v['vol']*100:.2f}%)", edgecolors='black', linewidth=1)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('夏普比率', fontsize=12)

ax.set_title('蒙特卡洛模拟 — 有效前沿', fontsize=14, fontweight='bold')
ax.set_xlabel('年化波动率', fontsize=12)
ax.set_ylabel('年化收益率', fontsize=12)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'最优夏普组合权重: {best["weights"]}')
print(f'最优夏普: 收益={best["return"]*100:.2f}%, 波动={best["vol"]*100:.2f}%, 夏普={best["sharpe"]:.4f}')
print(f'最小波动组合权重: {min_v["weights"]}')

## 13. Markowitz 有效前沿 (解析解)

In [ ]:
# Markowitz 均值-方差优化
mvo = portfolio.mean_variance_optimization(returns_df)

frontier = mvo['frontier']
if frontier:
    frontier_ret = [p[0] for p in frontier]
    frontier_vol = [p[1] for p in frontier]

    fig, ax = plt.subplots(figsize=(12, 7))

    # 有效前沿
    ax.plot(frontier_vol, frontier_ret, 'b-', linewidth=2.5, label='有效前沿')

    # 最小波动
    mv = mvo['min_vol']
    ax.scatter(mv['vol'], mv['return'], c='blue', marker='D', s=200,
               label=f"最小波动 ({mv['vol']*100:.2f}%)", edgecolors='black', linewidth=1, zorder=5)

    # 最大夏普
    ms = mvo['max_sharpe']
    ax.scatter(ms['vol'], ms['return'], c='red', marker='*', s=300,
               label=f"最大夏普 (SR={ms['sharpe']:.2f})", edgecolors='black', linewidth=1, zorder=5)

    # 资本市场线 (CML)
    rf = 0.02
    if ms['vol'] > 0:
        cml_x = np.linspace(0, max(frontier_vol) * 1.2, 100)
        cml_y = rf + (ms['return'] - rf) / ms['vol'] * cml_x
        ax.plot(cml_x, cml_y, '--', color='gray', alpha=0.7, label='资本市场线 (CML)')

    ax.set_title('Markowitz 有效前沿', fontsize=14, fontweight='bold')
    ax.set_xlabel('年化波动率', fontsize=12)
    ax.set_ylabel('年化收益率', fontsize=12)
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

print(f'最大夏普组合: {ms["weights"]}')
print(f'最小波动组合: {mv["weights"]}')

## 14. 风险平价组合

In [ ]:
try:
    rp = portfolio.risk_parity_portfolio(returns_df)
    print('风险平价组合:')
    for k, v in rp['weights'].items():
        print(f'  {k}: {v*100:.1f}%')
    print(f'\n年化收益: {rp["return"]*100:.2f}%')
    print(f'年化波动: {rp["vol"]*100:.2f}%')
    print(f'夏普比率: {rp["sharpe"]:.4f}')
except Exception as e:
    print(f'风险平价计算失败: {e}')

## 15. 生成分析报告

自动生成 Markdown 格式的基金分析报告。

In [ ]:
md_report = reporter.generate_report(fund_code)

# 保存报告
os.makedirs('data/reports', exist_ok=True)
report_path = f'data/reports/{fund_code}_full_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(md_report)

print(f'报告已保存至: {report_path}')
from IPython.display import Markdown
Markdown(md_report)

## 16. 导出数据

将基金净值数据导出为 CSV。

In [ ]:
csv_path = reporter.export_to_csv(fund_code)
print(f'CSV 已保存至: {csv_path}')

# 同时保存基金指标卡片
card = analyzer.fund_report_card(fund_code)
card_df = pd.DataFrame([card]).T
card_df.columns = ['数值']
card_df

## 总结

通过本 Notebook，你已学会如何使用 FundAnalyzer 工具包：

1. **搜索基金** — 按名称/代码搜索 6000+ 公募基金
2. **获取数据** — 基金信息、历史净值、区间收益、持仓
3. **计算指标** — 年化收益、波动率、最大回撤、夏普比率等 10+ 指标
4. **风险分析** — VaR、CVaR、回撤分析、连续亏损
5. **组合优化** — 蒙特卡洛模拟、Markowitz 有效前沿、风险平价
6. **报告生成** — 一键导出 Markdown 分析报告

---
**风险提示**: 本工具仅供学习研究，不构成投资建议。过往业绩不代表未来表现。